# RobustModelMaker Benchmark Suite

This notebook runs three real scientific benchmarks to demonstrate that **RobustModelMaker (ROBUST)** achieves comparable predictive performance to a full-feature baseline while selecting a substantially smaller feature subset.

Each scenario:
1. Loads a real scientific dataset via BenchMake archetypal train/test split (adversarial by design -- conservative scores).
2. Fits RobustModelMaker on the training partition (bootstrap stability selection + nested CV).
3. Fits a full-feature nested-CV baseline on the same training partition with the same algorithm.
4. Applies a 25+ test statistical battery to the per-fold score vectors.
5. Classifies the outcome as **preserved** (no significant loss), **sig. better \***, or **sig. worse \***.

All three scenarios use **Random Forest (RF)** for both RobustModelMaker and the baseline, making the comparison a clean demonstration of stability selection applied to the same algorithm across all task types:
- **SECOM Manufacturing** -- binary classification (590 features, real NaNs)
- **Urban Land Cover** -- 9-class multiclass (147 aerial-image features)
- **Graphene Oxide Bulk** -- regression (~309 MD structural descriptors)

Run cells individually to inspect each scenario, or run all to see the cross-scenario summary table at the bottom.

## Setup

The cell below loads `benchmark_suite.py` as a module using `importlib` so that the `if __name__ == "__main__":` block does **not** execute automatically. Adjust `REPO_ROOT` if your directory layout differs.

In [3]:
import sys
import os
import importlib.util
import warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)

# ---- Path configuration ----
# Set REPO_ROOT to the directory containing RobustModelMaker.py.
# Uncomment and edit the line below if running from a different location.
# REPO_ROOT = Path(r"path_to_your_RobustModelMaker_directory")
REPO_ROOT = Path(r"C:\Users\Amanda\Favorites\Machine Learning\RobustModelMaker")
sys.path.insert(0, str(REPO_ROOT))

# ---- Load benchmark_suite as a module ----
# importlib prevents the __main__ block from running; each scenario is
# controlled individually in the cells below.
_bs_path = REPO_ROOT / "benchmarks" / "benchmark_suite.py"
if not _bs_path.exists():
    raise FileNotFoundError(f"benchmark_suite.py not found at {_bs_path}")

_spec = importlib.util.spec_from_file_location("benchmark_suite", str(_bs_path))
bs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(bs)

# ---- Convenience aliases ----
_load_secom            = bs._load_secom
_load_urban_land_cover = bs._load_urban_land_cover
_load_graphene_oxide   = bs._load_graphene_oxide
run_scenario           = bs.run_scenario
print_summary_table    = bs.print_summary_table
DataUnavailable        = bs.DataUnavailable
ROBUST_PARAMS          = bs.ROBUST_PARAMS

# ---- Global parameter summary ----
p = ROBUST_PARAMS
print("Benchmark suite loaded successfully.")
print(f"  outer_cv={p['outer_cv']}  inner_cv={p['inner_cv']}  "
      f"n_bootstrap={p['n_bootstrap']}  stability_threshold={p['stability_threshold']}  "
      f"n_iter={p['n_iter']}  random_state={p['random_state']}")

# Containers for cross-scenario summary (populated by the cells below)
result_secom    = None
result_urban    = None
result_graphene = None

CuPy not installed. Using NumPy for CPU computations.
Benchmark suite loaded successfully.
  outer_cv=10  inner_cv=5  n_bootstrap=25  stability_threshold=0.6  n_iter=10  random_state=42


---
## Benchmark 1: SECOM Manufacturing (binary classification)

| Property | Value |
|---|---|
| Dataset | SECOM semiconductor sensor data (UCI) |
| Samples x features | 1,567 x 590 |
| Task | Binary pass/fail (~7% failure rate) |
| Algorithm | Random Forest (RF) |
| Metric | AUC-ROC (higher = better) |
| Challenge | Heavy class imbalance, extensive real NaN values |

The dataset is split with BenchMake's archetypal partitioning (adversarial: train and test sets are maximally separated in feature space). RobustModelMaker runs nested CV on the training partition only; the held-out test set is not used during fitting.

In [5]:
# Load SECOM dataset (downloads from UCI if not cached; requires internet access)
try:
    ds_secom = _load_secom()
    print(f"SECOM loaded: {ds_secom.X.shape[0]} samples x {ds_secom.X.shape[1]} features")
    print(f"  Train: {ds_secom.X_train.shape[0]}  |  Test (held-out): {ds_secom.X_test.shape[0]}")
except DataUnavailable as e:
    print(f"SKIP: {e}")
    ds_secom = None

SECOM loaded: 1567 samples x 590 features
  Train: 1253  |  Test (held-out): 314


In [6]:
if ds_secom is not None:
    result_secom = run_scenario(ds_secom, verbose=True)
else:
    print("SECOM dataset unavailable -- skipped.")


>>> SECOM Manufacturing: running ROBUST(RF, binary) ...
    ROBUST done (301 features selected). Running baseline ...

  SCENARIO: SECOM Manufacturing
  Semiconductor process sensor data; 1567 samples x 590 features, binary pass/fail, ~7% failure rate, extensive real NaN values
  Algorithm : RF   Task : binary   Runtime : 1906.6s
  Legend: ROBUST = RobustModelMaker stability-selected subset  |  BL = Full-feature nested-CV baseline

                                         DATASET                                        
------------------------------------------------------------------------------------------
  Total : 1567 samples x 590 features  (BenchMake split: train=1253, held-out test=314)
  Classes (train) : 0: 1170   1: 83

                               FEATURE SELECTION COMPARISON                             
------------------------------------------------------------------------------------------
  Baseline (BL)             :   590 features   score = 0.6814 +/- 0.0527
  ROB

---
## Benchmark 2: Urban Land Cover (9-class multiclass)

| Property | Value |
|---|---|
| Dataset | Urban Land Cover (UCI) |
| Samples x features | 675 x 147 |
| Task | 9 urban land cover classes from aerial image segments |
| Algorithm | Random Forest (RF) |
| Metric | AUC-OVR weighted (higher = better) |
| Challenge | Multi-class structure; highly correlated spectral/texture descriptors |

Classification is segment-level (not pixel-level), so there is no spatial autocorrelation between samples. RobustModelMaker is expected to drop a large fraction of the 147 highly correlated imagery features without significant performance loss.

In [8]:
# Load Urban Land Cover dataset (downloads from UCI; requires internet access)
try:
    ds_urban = _load_urban_land_cover()
    print(f"Urban Land Cover loaded: {ds_urban.X.shape[0]} samples x {ds_urban.X.shape[1]} features")
    print(f"  Train: {ds_urban.X_train.shape[0]}  |  Test (held-out): {ds_urban.X_test.shape[0]}")
    print(f"  Classes: {sorted(ds_urban.y.unique())}")
except DataUnavailable as e:
    print(f"SKIP: {e}")
    ds_urban = None

Urban Land Cover loaded: 675 samples x 147 features
  Train: 540  |  Test (held-out): 135
  Classes: ['asphalt ', 'building ', 'car ', 'concrete ', 'grass ', 'pool ', 'shadow ', 'soil ', 'tree ']


In [9]:
if ds_urban is not None:
    result_urban = run_scenario(ds_urban, verbose=True)
else:
    print("Urban Land Cover dataset unavailable -- skipped.")


>>> Urban Land Cover: running ROBUST(RF, multiclass) ...
    ROBUST done (66 features selected). Running baseline ...

  SCENARIO: Urban Land Cover
  Aerial image segments; 675 samples x 147 spectral/texture features, 9 urban land cover classes, segment-level (no spatial autocorrelation)
  Algorithm : RF   Task : multiclass   Runtime : 693.3s
  Legend: ROBUST = RobustModelMaker stability-selected subset  |  BL = Full-feature nested-CV baseline

                                         DATASET                                        
------------------------------------------------------------------------------------------
  Total : 675 samples x 147 features  (BenchMake split: train=540, held-out test=135)
  Classes (train) : building : 97   grass : 94   tree : 92   concrete : 84   shadow : 50   asphalt : 45   car : 30   soil : 25   pool : 23

                               FEATURE SELECTION COMPARISON                             
-------------------------------------------------------

---
## Benchmark 3: Graphene Oxide Bulk (regression)

| Property | Value |
|---|---|
| Dataset | CSIRO Graphene Oxide Bulk (local CSV) |
| Samples x features | ~1,617 x ~309 (after NaN-column and constant-column drop) |
| Task | Regression: Formation energy (eV) from structural descriptors |
| Algorithm | Random Forest (RF) |
| Metric | RMSE in eV (lower = better; displayed as positive RMSE) |
| Challenge | Very high-dimensional descriptor space; sparse/correlated MD features |

RF importance scores (MDI variance reduction) are naturally non-uniform across correlated structural descriptors, giving bootstrap stability selection a discriminative frequency distribution without algorithm-specific tuning.

**File requirement:** `Graphene_Oxide_Bulk.csv` must be present in the `benchmarks/` directory. This is a CSIRO dataset and is not downloaded automatically, but more information can be obtained here: https://doi.org/10.25919/5e30b45f9852c 

In [11]:
# Load Graphene Oxide Bulk dataset (requires local CSV file)
try:
    ds_graphene = _load_graphene_oxide()
    print(f"Graphene Oxide loaded: {ds_graphene.X.shape[0]} samples x {ds_graphene.X.shape[1]} features")
    print(f"  Train: {ds_graphene.X_train.shape[0]}  |  Test (held-out): {ds_graphene.X_test.shape[0]}")
    import numpy as np
    y_arr = ds_graphene.y.to_numpy(dtype=float)
    print(f"  Formation_energy: min={y_arr.min():.2f} eV  max={y_arr.max():.2f} eV  mean={y_arr.mean():.2f} eV")
    print(f"  Dataset-specific overrides: {ds_graphene.robust_params_override}")
except DataUnavailable as e:
    print(f"SKIP: {e}")
    ds_graphene = None

Graphene Oxide loaded: 1617 samples x 309 features
  Train: 1293  |  Test (held-out): 324
  Formation_energy: min=-88.28 eV  max=-66.09 eV  mean=-80.42 eV
  Dataset-specific overrides: {}


In [12]:
if ds_graphene is not None:
    result_graphene = run_scenario(ds_graphene, verbose=True)
else:
    print("Graphene Oxide dataset unavailable -- skipped.")


>>> Graphene Oxide Bulk: running ROBUST(RF, regression) ...
    ROBUST done (150 features selected). Running baseline ...

  SCENARIO: Graphene Oxide Bulk
  MD-derived structural descriptors; 1617 samples x 309 features, regression target = Formation_energy (eV), 19 stoichiometries
  Algorithm : RF   Task : regression   Runtime : 12116.0s
  Legend: ROBUST = RobustModelMaker stability-selected subset  |  BL = Full-feature nested-CV baseline

                                         DATASET                                        
------------------------------------------------------------------------------------------
  Total : 1617 samples x 309 features  (BenchMake split: train=1293, held-out test=324)
  Target : continuous   mean=-80.367   std=5.551

                               FEATURE SELECTION COMPARISON                             
------------------------------------------------------------------------------------------
  Baseline (BL)             :   309 features   RMSE (low

---
## Cross-scenario summary

The table below consolidates the key figures from all three benchmarks: feature counts, scores, and statistical outcome.

- Regression rows show **RMSE** (positive, lower is better).
- Classification rows show **AUC** (higher is better).
- **+delta** is positive when RobustModelMaker is better than baseline in each metric's own convention.
- **Outcome** is determined by a paired Wilcoxon (or paired t-test) on the per-fold score vectors, not by the point estimate alone.

> **Note on BenchMake splits:** All benchmarks use adversarial archetypal splits that maximise the distance between train and test sets in feature space. This makes scores conservative relative to conventional random splits. The RobustModelMaker vs. baseline comparison is internally consistent because both models use the same split.

In [14]:
all_results = [r for r in [result_secom, result_urban, result_graphene] if r is not None]

if all_results:
    print_summary_table(all_results)
    total = sum(r["elapsed"] for r in all_results)
    print(f"  Total wall-clock time: {total:.1f}s")
else:
    print("No results to summarise -- all datasets were unavailable or no benchmarks have been run yet.")


  CROSS-SCENARIO SUMMARY
  Legend: ROBUST = RobustModelMaker stability-selected subset  |  BL = Full-feature nested-CV baseline
  Goal: feature reduction while preserving predictive performance.
  Outcome: 'preserved' = no significant difference from full-feature baseline (p >= 0.05) -- primary success criterion
           'sig. worse *' = significant performance cost (p < 0.05)  |  'sig. better *' = unexpected improvement
------------------------------------------------------------------------------------------------------------------------------
  Scenario                 Task          n_train x p  ROBUST feats  Red%   BL score ROBUST score   +delta    p-val Outcome    
  ------------------------ ----------- -------------  ------------ -----  --------- ------------ --------  ------- -----------
  (regression rows show RMSE -- lower is better; classification rows show AUC -- higher is better)

  SECOM Manufacturing      binary        1253 x  590           301    49%     0.6814       